# Mistral-7B-Instruct-v0.2 → Text-to-SQL con LoRA (PEFT) 
**Objetivo:** usar 5000 registros de `gretelai/synthetic_text_to_sql` para entrenar LoRA y generar SQL desde NL.



### Flujo de trabajo

1. Instalar dependencias

2. Imports y chequeos

3. Cargar dataset (5000 registros)

4. División en train / eval / test (80/20 y luego 80/20 dentro de train) ✅

5. Tokenizer y chequeos del token

6. Función de tokenización (map sobre cada split)

7. Cargar modelo

8. Aplicar LoRA (PEFT)

9. Configurar entrenamiento (Trainer recibe train_dataset y eval_dataset)

10. Entrenamiento

11. Graficar training loss

12. Evaluación final con test_dataset

13. Guardar

## Instalar dependencias

In [ ]:
# =========================================
# Celda 1 - Instalación de dependencias
# =========================================
# Instalamos versiones compatibles para LoRA + PEFT + Mistral + SFTTrainer
!pip install transformers peft accelerate bitsandbytes datasets sentencepiece safetensors torch trl

print("✅ Dependencias instaladas (incluyendo trl para SFTTrainer)")

## Imports y chequeos

In [ ]:
# Imports principales
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer  # 🚀 Agregar SFTTrainer como en Llama
import torch
from torch.utils.data import DataLoader
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import login


import bitsandbytes as bnb 
# --- Fix: desactivar memory efficient backward ---
bnb.nn.Linear8bitLt.use_memory_efficient_backward = False
bnb.nn.Linear8bitLt.use_bias = True

# Chequeos rápidos
print("CUDA available?", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise SystemError("No GPU detectada. Activa GPU en la configuración de Kaggle.")

# 🚀 Mostrar info de GPUs como en Llama
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"🎮 GPU {i}: {gpu_name}")
        print(f"   VRAM: {gpu_memory:.1f} GB")

## Autenticación en Hugging Face

In [ ]:
# Login en Hugging Face (necesario para Llama)
login()

print("✅ Autenticado en Hugging Face")

## Cargar dataset

In [ ]:
# Cargar dataset desde Hugging Face
ds = load_dataset("gretelai/synthetic_text_to_sql")

# Revisar splits disponibles
print(ds)

# Vamos a usar el split 'train' (si no existe, usamos el primero disponible)
split_name = "train" if "train" in ds else list(ds.keys())[0]
print("Usando split:", split_name)

# Tomamos 5000 muestras aleatorias (seed para reproducibilidad)
num_samples = 5000
seed = 42

ds_small = ds[split_name].shuffle(seed=seed).select(range(min(num_samples, len(ds[split_name]))))

print("Tamaño seleccionado:", len(ds_small))
# Mostrar un ejemplo
print(ds_small[0])


## Dividir en train / eval / test
Del dataset original se toman 5000 registros y se divide en:
* 80% train y 20% test

Luego el conjunto de train se divide en:
* 80% train y 20% eval

In [ ]:
# Paso 1: dividir en 80% train_base y 20% test
split_1 = ds_small.train_test_split(test_size=0.2, seed=seed)
train_base = split_1["train"]
test_dataset = split_1["test"]

# Paso 2: del train_base, dividir en 80% train y 20% eval
split_2 = train_base.train_test_split(test_size=0.2, seed=seed)
train_dataset = split_2["train"]
eval_dataset = split_2["test"]

# Chequear tamaños finales
print(f"Train: {len(train_dataset)} registros")
print(f"Eval: {len(eval_dataset)} registros")
print(f"Test: {len(test_dataset)} registros")


### Tokenizer
Esto asegura que el pad_token exista (si no, da error en el entrenamiento).

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Si el tokenizer no tiene pad_token definido, lo añadimos
if tokenizer.pad_token_id is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

print("Tokenizer loaded. Vocab size:", tokenizer.vocab_size)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)


## Tokenización simplificada (estilo Llama)

### 💡 Alternativa: Enfoque simplificado como Llama (sin -100)

In [ ]:
# Max length y función de prompt
MAX_LENGTH = 512

def make_prompt(sql_prompt):
    """Crear prompt estilo instruction-following para Mistral"""
    return f"### Instruction:\n{sql_prompt}\n\n### Response:\n"

In [ ]:
# 🚀 TOKENIZACIÓN SIMPLIFICADA ESTILO LLAMA
# En lugar de usar -100 para enmascarar, usamos el texto completo como labels

def tokenize_example_simple(example):
    """
    Enfoque simplificado: el input y los labels son idénticos
    - No hay enmascaramiento de prompt (-100)
    - El modelo aprende de toda la secuencia
    - Más simple y menos propenso a errores
    """
    # Crear el texto completo
    prompt_text = make_prompt(example["sql_prompt"])
    target_text = example["sql"]
    full_text = prompt_text + target_text
    
    # Tokenizar todo junto
    encoding = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )
    
    # Los labels son idénticos a los input_ids
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": encoding["input_ids"].copy()  # ⭐ ¡Simple!
    }

# Probar con un ejemplo
print("🧪 PRUEBA DE TOKENIZACIÓN SIMPLE:")
test_example = train_dataset[0]
result_simple = tokenize_example_simple(test_example)

print(f"Input IDs length: {len(result_simple['input_ids'])}")
print(f"Labels length: {len(result_simple['labels'])}")
print(f"Son idénticos: {result_simple['input_ids'] == result_simple['labels']}")

# Verificar que no hay -100 en los labels
labels_count = len([l for l in result_simple['labels'] if l != -100])
print(f"Labels válidos (no -100): {labels_count}/{len(result_simple['labels'])}")

In [ ]:
# 🔄 PREPARAR DATOS PARA SFTTrainer
print("Preparando datos para SFTTrainer...")

# SFTTrainer espera un campo 'text' con el texto completo
def prepare_for_sft(example):
    """
    Preparar ejemplo para SFTTrainer
    SFTTrainer maneja la tokenización internamente
    """
    prompt_text = make_prompt(example["sql_prompt"])
    target_text = example["sql"]
    full_text = prompt_text + target_text
    
    return {"text": full_text}

# Aplicar transformación para SFTTrainer
train_dataset_sft = train_dataset.map(prepare_for_sft, remove_columns=train_dataset.column_names)
eval_dataset_sft = eval_dataset.map(prepare_for_sft, remove_columns=eval_dataset.column_names)

print(f"✅ Train SFT: {len(train_dataset_sft)} ejemplos")
print(f"✅ Eval SFT: {len(eval_dataset_sft)} ejemplos")

# Verificación rápida
print("\n🔍 VERIFICACIÓN SFT:")
example = train_dataset_sft[0]
print(f"Campos disponibles: {list(example.keys())}")
print(f"Texto de ejemplo:\n{example['text'][:200]}...")

# También mantener los datasets tokenizados por si acaso
train_dataset_simple = train_dataset.map(tokenize_example_simple, remove_columns=train_dataset.column_names)
eval_dataset_simple = eval_dataset.map(tokenize_example_simple, remove_columns=eval_dataset.column_names)
test_dataset_simple = test_dataset.map(tokenize_example_simple, remove_columns=test_dataset.column_names)

### 🆚 Comparación: Método complejo (-100) vs Simple (Llama)

**Método complejo con -100:**
- ✅ Más "teóricamente correcto" 
- ❌ Complejo de implementar
- ❌ Propenso a errores (como acabamos de ver)
- ❌ Puede no entrenar si todos los labels son -100

**Método simple (estilo Llama):**
- ✅ Muy simple de implementar
- ✅ Siempre funciona (sin labels vacíos)
- ✅ Usado exitosamente en muchos modelos
- ⚠️ El modelo aprende también del prompt (no solo de la respuesta)

**🎯 Recomendación:** Para text-to-SQL, el método simple es perfectamente válido y mucho más confiable.

## Cargar modelo
Sin usar cuantizacion

In [ ]:
# --- Modelo ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # ⚡ usa FP16 para eficiencia
    device_map="auto",          # 🚀 Distribución automática como en Llama
    trust_remote_code=True,
)

# Ajustar config para que use el pad_token recién creado
model.config.pad_token_id = tokenizer.pad_token_id

# Redimensionar embeddings si agregamos tokens nuevos
model.resize_token_embeddings(len(tokenizer))

print("✅ Modelo cargado sin cuantización")
print(f"🚀 Distribución automática de GPUs habilitada")

# Verificar distribución del modelo
if torch.cuda.device_count() > 1:
    print(f"🎮 Detectadas {torch.cuda.device_count()} GPUs")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"   Primer parámetro entrenable en: {param.device}")
            break

## Aplicar LoRA (PEFT)

In [ ]:
# --- Configuración de LoRA ---
lora_config = LoraConfig(
    r=16,              # rank de las matrices de bajo rango
    lora_alpha=32,     # factor de escala
    target_modules=["q_proj", "v_proj"],  # capas donde aplicar LoRA (típico en LLMs)
    lora_dropout=0.05, # dropout durante fine-tuning
    bias="none",
    task_type="CAUSAL_LM"
)

# --- Aplicar LoRA al modelo ---
model = get_peft_model(model, lora_config)

# Mostrar cuántos parámetros se entrenan
model.print_trainable_parameters()


## Configuracion del trainer

In [ ]:
# --- Callback para guardar losses ---
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        # Guardar training loss
        if logs is not None and "loss" in logs:
            self.train_losses.append(logs["loss"])
        # Guardar eval loss si existe
        if logs is not None and "eval_loss" in logs:
            self.eval_losses.append(logs["eval_loss"])

# --- Callback instancia ---
loss_logger = LossLoggerCallback()

# 🚀 Configuración como en Llama: optimizada para múltiples GPUs
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",
    overwrite_output_dir=True,
    
    # Configuración básica
    num_train_epochs=1,
    per_device_train_batch_size=2,          # Ajustar según GPUs disponibles
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,          # Para simular batch más grande
    
    # Optimización para múltiples GPUs
    learning_rate=2e-5,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    
    # Guardado y evaluación
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    
    # Optimización como en Llama
    optim="adamw_8bit",                     # 🔥 Optimizador eficiente
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precisión
    fp16=torch.cuda.is_available(),         # 🔥 FP16 automático si hay GPU
    bf16=False,
    
    # Otros
    dataloader_drop_last=True,
    remove_unused_columns=False,            # 🚀 Importante para SFTTrainer
    report_to="none",
    seed=42,
)

# 🚀 Usar SFTTrainer con configuración mínima
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_sft,        # 🔥 Usar datasets preparados para SFT
    eval_dataset=eval_dataset_sft,          # 🔥 Usar datasets preparados para SFT
    args=training_args,
)

# 🔥 Configurar tokenizer DESPUÉS de crear el trainer
trainer.tokenizer = tokenizer
if trainer.tokenizer.pad_token is None:
    trainer.tokenizer.pad_token = trainer.tokenizer.eos_token

# Agregar callback
trainer.add_callback(loss_logger)

print("✅ SFTTrainer configurado correctamente")
print("🚀 Optimizado para múltiples GPUs")
print(f"📦 Train dataset: {len(trainer.train_dataset)} ejemplos")
print(f"📦 Eval dataset: {len(trainer.eval_dataset)} ejemplos")

## Entrenamiento

In [ ]:
# --- Entrenamiento ---
train_result = trainer.train()
print("✅ Entrenamiento finalizado")



## Graficar training loss y validation loss

In [ ]:
# --- Graficar training y eval loss ---
plt.figure(figsize=(8,5))
plt.plot(loss_logger.train_losses, label="Training Loss")
if loss_logger.eval_losses:
    plt.plot(range(0, len(loss_logger.train_losses), int(len(loss_logger.train_losses)/len(loss_logger.eval_losses))),
             loss_logger.eval_losses, label="Eval Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training & Eval Loss")
plt.legend()
plt.show()

# --- Detectar overfitting simple ---
if loss_logger.eval_losses and min(loss_logger.eval_losses) > min(loss_logger.train_losses):
    print("⚠️ Posible overfitting: eval_loss > train_loss")
else:
    print("✅ Sin indicios claros de overfitting por ahora")

## Guardar modelo

In [ ]:
trainer.save_model("./trained_model")  
tokenizer.save_pretrained("./trained_model")   

# el modelo y el tokenizer son un par inseparable. El modelo aprendió con ese tokenizer, así que si no 
# se guarda, después no se puede reproducir los mismos resultados.

## Evaluar modelo entrenado con el dataset de TEST

In [ ]:
import json
from tqdm import tqdm

# Poner modelo en modo evaluación
model.eval()

results = []

for i in tqdm(range(len(eval_dataset_tok))):
    example = test_dataset_tok[i]
    prompt = test_dataset[i]["sql_prompt"]
    expected_sql = test_dataset[i]["sql"]

    # Preparar input
    #input_ids = torch.tensor([example["input_ids"]]).to(model.device)
    # Preparar input con atención
    input_ids = torch.tensor([example["input_ids"]]).to(model.device)
    attention_mask = torch.tensor([example["attention_mask"]]).to(model.device)

    # Generar predicción
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=False  # greedy decoding
        )

    # Decodificar
    predicted_sql = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Guardar resultado
    results.append({
        "index": i,
        "prompt": prompt,
        "expected_sql": expected_sql,
        "predicted_sql": predicted_sql
    })

# Guardar en JSON
with open("test_predictions_trained.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Guardado en eval_predictions_trained.json con {len(results)} predicciones")

NameError: name 'model' is not defined

## Liberar memoria

del trainer → elimina el objeto Trainer, que a su vez contiene el modelo (trainer.model), optimizadores, etc. Esto libera las referencias en Python.

torch.cuda.empty_cache() → le avisa a PyTorch que libere la memoria de GPU que estaba usando para ese modelo.

Resultado: la GPU queda libre y ya podés cargar otro modelo

In [ ]:
del trainer
torch.cuda.empty_cache()

## Evaluar modelo original con el dataset de TEST

In [ ]:
# Cargar modelo original
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16)


In [ ]:
# Poner modelo en modo evaluación
base_model.eval()

results = []

for i in tqdm(range(len(eval_dataset_tok))):
    example = test_dataset_tok[i]
    prompt = test_dataset[i]["sql_prompt"]
    expected_sql = test_dataset[i]["sql"]

    # Preparar input
    input_ids = torch.tensor([example["input_ids"]]).to(base_model.device)

    # Generar predicción
    with torch.no_grad():
        output_ids = base_model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=False  # greedy decoding
        )

    # Decodificar
    predicted_sql = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Guardar resultado
    results.append({
        "index": i,
        "prompt": prompt,
        "expected_sql": expected_sql,
        "predicted_sql": predicted_sql
    })

# Guardar en JSON
with open("test_predictions_original.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Guardado en eval_predictions_original.json con {len(results)} predicciones")

--------------------------------------------------------

In [ ]:
# Cargar modelo entrenado
model.eval()  # poner en modo evaluación

example = eval_dataset_tok[0]  # primer ejemplo de eval
input_ids = torch.tensor([example["input_ids"]]).to(model.device)

# Generar predicción
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=128,  # ajusta según el largo de tu SQL
        do_sample=False      # greedy decoding; True si querés muestreo
    )

# Decodificar
predicted_sql = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("NL prompt:", eval_dataset[0]["sql_prompt"])
print("Predicted SQL:", predicted_sql)
print("Expected SQL:", eval_dataset[0]["sql"])


In [ ]:
for i in range(5):  # probar los primeros 5 ejemplos
    example = eval_dataset_tok[i]
    input_ids = torch.tensor([example["input_ids"]]).to(model.device)
    output_ids = model.generate(input_ids, max_new_tokens=128, do_sample=False)
    predicted_sql = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Prompt: {eval_dataset[i]['sql_prompt']}")
    print(f"Predicted: {predicted_sql}")
    print(f"Expected: {eval_dataset[i]['sql']}")
    print("-"*50)
